<a href="https://colab.research.google.com/github/pacekonathyo/EW-DDBS_for_ctg_ctu/blob/main/02_PFR2025_P_1_CTU_UHB__patched_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# EW-DDBS — External Validation on CTU-UHB

This notebook mirrors the methodology of:
- 1D-CNN topology guide (Conv1D-64 → Conv1D-32 → GAP → Dense-2)
- Full EW-DDBS with 4 ablation variants (Entropy/Safe × ±Tomek)
- 10-fold stratified CV with leakage-controlled preprocessing
- Hyperparameter tuning per classifier (15 classifiers, light grids)
- AUPRC + AUC + F1-Macro + G-Mean + Balanced Accuracy
- Friedman + Nemenyi + Wilcoxon ablation tests

**Expected runtime on Colab T4: 60–90 minutes.**

Records and pH labels are fetched directly from PhysioNet via `wfdb` — no external CSV needed.

In [ ]:
# 1. Install library WFDB bawaan PhysioNet
!pip install wfdb

import wfdb
import pandas as pd

# 2. Akses record data langsung dari server PhysioNet (contoh membaca record pasien '1001')
# pn_dir adalah direktori resmi dataset CTU-UHB di PhysioNet
record = wfdb.rdrecord('1001', pn_dir='ctu-uhb-ctgdb/1.0.0')

# 3. Ekstrak sinyal FHR (Fetal Heart Rate) dan UC (Uterine Contraction)
df_signals = record.to_dataframe()

print("Fitur yang tersedia:", record.sig_name)
print("Durasi rekaman:", record.sig_len, "titik data")
print("\nCuplikan Data:")
print(df_signals.head())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 53.9 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but y

In [ ]:
!pip install wfdb scikit-posthocs imbalanced-learn xgboost
!pip install scikit-posthocs
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


In [ ]:
# Di awal notebook
%tensorflow_version 2.x
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Main experiment
Defines feature extraction, CNN topology guide, EW-DDBS variants, classifier zoo with tuning, and the 10-fold evaluation loop. Runs the experiment at the bottom of the cell.

In [ ]:
# -*- coding: utf-8 -*-
"""
EW-DDBS External Validation on CTU-UHB
- Methodologically CONSISTENT with the UCI experiment (Notebook 01)
- Adds: AUPRC, full 4-variant ablation, hyperparameter tuning, Friedman/Nemenyi
- Runtime ~ 60-90 min on Colab T4 GPU
"""

import os, time, warnings
import numpy as np
import pandas as pd
import wfdb
from scipy import stats
from scipy.signal import welch

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (f1_score, balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, average_precision_score)
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.base import clone
from sklearn.utils.class_weight import compute_class_weight
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import (SMOTE, BorderlineSMOTE, ADASYN,
                                    SVMSMOTE, KMeansSMOTE, RandomOverSampler)
from imblearn.under_sampling import TomekLinks

from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              BaggingClassifier, ExtraTreesClassifier, AdaBoostClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Hyperparameters - MIRRORED FROM UCI EXPERIMENT (Notebook 01)
T_TEMP = 0.5          # entropy temperature
TAU = 0.2             # safe-region threshold
K_NN = 5              # nearest neighbours
ETA_JITTER = 0.05     # jitter scale
SAMPLING_STRATEGY_TOMEK = 'all'

# =====================================================================
# 1. FEATURE EXTRACTION FROM RAW CTU-UHB SIGNALS
# =====================================================================
def extract_ctg_features(signal_array):
    """Statistical features from a 1-D CTG signal (FHR or UC channel)."""
    clean_signal = signal_array[~np.isnan(signal_array)]
    if len(clean_signal) < 10:
        return [0]*15
    mean_val   = np.mean(clean_signal)
    std_val    = np.std(clean_signal)
    median_val = np.median(clean_signal)
    min_val    = np.min(clean_signal)
    max_val    = np.max(clean_signal)
    range_val  = max_val - min_val
    q25, q75   = np.percentile(clean_signal, [25, 75])
    iqr        = q75 - q25
    diff_signal = np.diff(clean_signal)
    stv = np.mean(np.abs(diff_signal))                   # short-term variation
    if len(clean_signal) > 32:
        freqs, psd = welch(clean_signal, fs=4, nperseg=min(32, len(clean_signal)))
        lf_power = np.trapz(psd[(freqs >= 0.03) & (freqs < 0.15)]) if len(psd) > 0 else 0
        hf_power = np.trapz(psd[(freqs >= 0.15) & (freqs < 0.5)])  if len(psd) > 0 else 0
        lf_hf_ratio = lf_power / (hf_power + 1e-10)
    else:
        lf_hf_ratio = 0
    hist, _ = np.histogram(clean_signal, bins=10, density=True)
    hist = hist[hist > 0]
    entropy = -np.sum(hist * np.log(hist + 1e-10))
    return [mean_val, std_val, median_val, min_val, max_val, range_val,
            q25, q75, iqr, stv, lf_hf_ratio, entropy,
            stats.skew(clean_signal), stats.kurtosis(clean_signal),
            np.percentile(clean_signal, 90)]


def parse_ph_from_header(header_comments):
    """
    Extract pH value from CTU-UHB record header comments.
    The .hea file contains lines like '#pH      7.25' (with leading '#' optional).
    """
    for line in header_comments:
        # comments come with '#' stripped by wfdb but format is e.g. 'pH      7.25'
        s = line.strip().lstrip('#').strip()
        parts = s.split()
        if len(parts) >= 2 and parts[0].lower() == 'ph':
            try:
                return float(parts[1])
            except ValueError:
                continue
    return None


def prepare_ctu_uhb(max_records=552):
    """
    Fetch CTU-UHB records DIRECTLY from PhysioNet.
    No external metadata CSV needed - pH is read from each record header.
    Caches result to .npz so subsequent runs are instant.
    """
    cache_file = 'ctu_uhb_features_cache.npz'
    if os.path.exists(cache_file):
        print("Loading from cache (ctu_uhb_features_cache.npz)...")
        data = np.load(cache_file)
        X, y = data['X'], data['y']
        print(f"  -> {X.shape[0]} samples, {X.shape[1]} features")
        print(f"  -> class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
        return X, y

    print("Fetching record list from PhysioNet (ctu-uhb-ctgdb/1.0.0)...")
    record_ids = wfdb.get_record_list('ctu-uhb-ctgdb/1.0.0')
    print(f"  -> {len(record_ids)} records available")

    feats, labels = [], []
    n_skipped_no_ph, n_skipped_error = 0, 0

    print(f"\nDownloading & extracting features (target: {max_records} records)...")
    for i, rid in enumerate(record_ids):
        if len(feats) >= max_records:
            break
        try:
            # Read header first to get pH (fast: tiny file)
            hdr = wfdb.rdheader(rid, pn_dir='ctu-uhb-ctgdb/1.0.0')
            ph_val = parse_ph_from_header(hdr.comments)
            if ph_val is None:
                n_skipped_no_ph += 1
                continue

            # Then read signal
            rec = wfdb.rdrecord(rid, pn_dir='ctu-uhb-ctgdb/1.0.0')
            fhr = rec.p_signal[:, 0]
            uc  = rec.p_signal[:, 1]
            feats.append(extract_ctg_features(fhr) + extract_ctg_features(uc))
            labels.append(0 if ph_val >= 7.15 else 1)  # binary: Normal vs Pathological

            if (len(feats)) % 50 == 0:
                print(f"  ...{len(feats)}/{max_records} processed "
                      f"(scanned {i+1}/{len(record_ids)})")
        except Exception as e:
            n_skipped_error += 1
            if n_skipped_error <= 3:
                print(f"  skipped {rid}: {type(e).__name__}: {str(e)[:80]}")
            continue

    X = np.array(feats, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)
    np.savez_compressed(cache_file, X=X, y=y)
    print(f"\nDataset ready: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
    if n_skipped_no_ph:  print(f"  ({n_skipped_no_ph} records had no parseable pH)")
    if n_skipped_error:  print(f"  ({n_skipped_error} records failed to download)")
    return X, y


# =====================================================================
# 2. CNN TOPOLOGY GUIDE  (matches UCI: Conv1D-64 + Conv1D-32 + GAP)
# =====================================================================
def get_cnn_guidance(X_train, y_train):
    n_features = X_train.shape[1]
    n_classes  = len(np.unique(y_train))

    inputs = layers.Input(shape=(n_features, 1))
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(32, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)                # 32-D latent vector
    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy')

    # Class-balanced training (same as UCI)
    cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw_dict = {c: w for c, w in zip(np.unique(y_train), cw)}

    X_cnn = X_train.reshape(-1, n_features, 1)
    es = callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
    model.fit(X_cnn, y_train, epochs=100, batch_size=32,
              class_weight=cw_dict, verbose=0, callbacks=[es])

    probs = model.predict(X_cnn, verbose=0)
    entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1)
    extractor = models.Model(inputs, model.layers[-2].output)
    deep_feats = extractor.predict(X_cnn, verbose=0)
    return entropy, deep_feats


# =====================================================================
# 3. EW-DDBS WITH 4 VARIANTS  (matches UCI implementation)
# =====================================================================
def compute_weighted_safe_sampling(X, y, entropy, deep_feats,
                                   use_safe=True, use_entropy=True,
                                   T=T_TEMP, tau=TAU, k=K_NN, eta=ETA_JITTER):
    """
    Compute sampling weights and synthesise minority samples.
    use_safe    : apply Safe-Ratio gating
    use_entropy : apply exp(H/T) entropy weighting
    Returns balanced (X_res, y_res).
    """
    X_res, y_res = X.copy(), y.copy()
    classes, counts = np.unique(y, return_counts=True)
    max_count = max(counts)

    # Per-class variance for jitter (diagonal covariance, computed on minority subset)
    for cls in classes:
        mask = y == cls
        if np.sum(mask) >= max_count:
            continue

        num_add  = max_count - np.sum(mask)
        idx_cls  = np.where(mask)[0]
        if len(idx_cls) < 2:
            chosen = np.random.choice(idx_cls, size=num_add, replace=True)
            X_res = np.vstack([X_res, X[chosen]])
            y_res = np.hstack([y_res, [cls]*num_add])
            continue

        # 1) Safe-Ratio in latent space
        k_eff = min(k, len(idx_cls)-1)
        nn_lat = NearestNeighbors(n_neighbors=k_eff+1).fit(deep_feats)
        _, ind_lat = nn_lat.kneighbors(deep_feats[idx_cls])
        safe_ratio = np.mean(y[ind_lat[:, 1:]] == cls, axis=1)

        # 2) Build weights
        if use_entropy:
            w_ent = np.exp(entropy[idx_cls] / T)
        else:
            w_ent = np.ones(len(idx_cls))
        if use_safe:
            w_safe = safe_ratio.copy()
            w_safe[safe_ratio < tau] = 0.0
            weights = w_ent * w_safe
        else:
            weights = w_ent

        if weights.sum() <= 0:
            weights = np.ones_like(weights)
        prob = weights / weights.sum()

        # 3) Synthesise in input space (latent-guided)
        nn_in = NearestNeighbors(n_neighbors=k_eff+1).fit(X[idx_cls])
        sigma = np.diag(np.var(X[idx_cls], axis=0)) * eta
        parent_local = np.random.choice(len(idx_cls), size=num_add, p=prob)
        _, ind_in = nn_in.kneighbors(X[idx_cls][parent_local])
        neighbour_local = ind_in[:, 1]

        new_samples = []
        for i in range(num_add):
            xp = X[idx_cls][parent_local[i]]
            xn = X[idx_cls][neighbour_local[i]]
            lam = np.random.uniform(0.1, 0.9)
            delta = np.random.multivariate_normal(np.zeros(X.shape[1]), sigma)
            new_samples.append(xp + lam*(xn - xp) + delta)
        new_samples = np.array(new_samples)
        X_res = np.vstack([X_res, new_samples])
        y_res = np.hstack([y_res, [cls]*num_add])

    return X_res, y_res


def run_ew_ddbs(X, y, entropy, deep_feats, use_safe, use_entropy, apply_tomek):
    """Full EW-DDBS pipeline: weighted sampling + optional Tomek cleaning."""
    X_res, y_res = compute_weighted_safe_sampling(
        X, y, entropy, deep_feats,
        use_safe=use_safe, use_entropy=use_entropy)
    if apply_tomek:
        try:
            tl = TomekLinks(sampling_strategy=SAMPLING_STRATEGY_TOMEK)
            X_res, y_res = tl.fit_resample(X_res, y_res)
        except Exception:
            pass
    return X_res, y_res


# =====================================================================
# 4. CLASSIFIERS  (15, with hyperparameter tuning - matches UCI)
# =====================================================================
def get_classifier_with_tuning(name, X_train, y_train):
    """Return a (lightly) tuned classifier. Minimal grids to keep runtime tractable."""
    base = {
        "GBM":       (GradientBoostingClassifier(random_state=SEED),
                      {'n_estimators': [100, 200]}),
        "RF":        (RandomForestClassifier(class_weight='balanced',
                                             random_state=SEED, n_jobs=-1),
                      {'n_estimators': [100, 200]}),
        "Bagging":   (BaggingClassifier(random_state=SEED, n_jobs=-1),
                      {'n_estimators': [10, 30]}),
        "XGBoost":   (XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                                    random_state=SEED, n_jobs=-1, verbosity=0),
                      {'n_estimators': [100, 200]}),
        "ExtraTrees":(ExtraTreesClassifier(class_weight='balanced',
                                           random_state=SEED, n_jobs=-1),
                      {'n_estimators': [100, 200]}),
        "AdaBoost":  (AdaBoostClassifier(random_state=SEED),
                      {'n_estimators': [50, 100]}),
        "DT":        (DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
                      {'max_depth': [None, 10]}),
        "SVM-RBF":   (SVC(kernel='rbf', probability=True, random_state=SEED),
                      {'C': [1, 10]}),
        "KNN":       (KNeighborsClassifier(n_jobs=-1),
                      {'n_neighbors': [3, 5]}),
        "SGD":       (SGDClassifier(random_state=SEED, n_jobs=-1, loss='log_loss'),
                      {'alpha': [1e-4, 1e-3]}),
        "LinearSVM": (LinearSVC(random_state=SEED, max_iter=2000),
                      {'C': [1, 10]}),
        "LogReg":    (LogisticRegression(random_state=SEED, n_jobs=-1, max_iter=2000),
                      {'C': [1, 10]}),
        "LDA":       (LinearDiscriminantAnalysis(), {}),
        "QDA":       (QuadraticDiscriminantAnalysis(), {}),
        "GNB":       (GaussianNB(), {}),
    }
    clf, grid = base[name]
    if grid:
        gs = GridSearchCV(clf, grid, cv=3, n_jobs=-1, scoring='f1_macro')
        gs.fit(X_train, y_train)
        return gs.best_estimator_
    return clone(clf).fit(X_train, y_train)


# =====================================================================
# 5. EVALUATION  (F1, G-Mean, BalAcc, AUC, AUPRC - matches UCI)
# =====================================================================
def g_mean_multiclass(y_true, y_pred):
    """Multi-class G-Mean: ((prod_c sqrt(Sens_c * Spec_c)))^(1/C)."""
    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(divide='ignore', invalid='ignore'):
        sens = np.diag(cm) / np.sum(cm, axis=1)
        spec = (np.sum(cm) - np.sum(cm, axis=1) - np.sum(cm, axis=0)
                + np.diag(cm)) / (np.sum(cm) - np.sum(cm, axis=1) + 1e-10)
        g = np.sqrt(sens * spec)
    return float(np.nanmean(g))


def evaluate_model(y_true, y_pred, y_proba=None):
    m = {
        'F1-Macro':         f1_score(y_true, y_pred, average='macro'),
        'BalancedAccuracy': balanced_accuracy_score(y_true, y_pred),
        'G-Mean':           g_mean_multiclass(y_true, y_pred),
    }
    if y_proba is not None:
        y_bin = label_binarize(y_true, classes=np.unique(y_true))
        if y_bin.shape[1] == 1:
            m['AUC']   = roc_auc_score(y_true, y_proba[:, 1])
            m['AUPRC'] = average_precision_score(y_true, y_proba[:, 1])
        else:
            m['AUC']   = roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro')
            m['AUPRC'] = average_precision_score(y_bin, y_proba, average='macro')
    else:
        m['AUC']   = np.nan
        m['AUPRC'] = np.nan
    return m


def predict_proba_safe(clf, X):
    """Get probabilities, falling back to decision_function for SVMs/SGD when needed."""
    if hasattr(clf, 'predict_proba'):
        try:
            return clf.predict_proba(X)
        except Exception:
            pass
    if hasattr(clf, 'decision_function'):
        try:
            dec = clf.decision_function(X)
            if dec.ndim == 1:
                # Binary: convert decision values to a 2-column proba via sigmoid
                p1 = 1.0 / (1.0 + np.exp(-dec))
                return np.column_stack([1 - p1, p1])
            else:
                e = np.exp(dec - dec.max(axis=1, keepdims=True))
                return e / e.sum(axis=1, keepdims=True)
        except Exception:
            return None
    return None


# =====================================================================
# 6. MAIN EXPERIMENT
# =====================================================================
def main_experiment(X, y, n_splits=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    # Preprocessing (impute + scale + PCA-95%) applied on train fold only
    samplers_classical = {
        "Baseline":          None,
        "RandomOverSampler": RandomOverSampler(random_state=SEED),
        "SMOTE":             SMOTE(random_state=SEED, k_neighbors=3),
        "BorderlineSMOTE":   BorderlineSMOTE(random_state=SEED, k_neighbors=3),
        "ADASYN":            ADASYN(random_state=SEED, n_neighbors=3),
        "SVMSMOTE":          SVMSMOTE(random_state=SEED, k_neighbors=3),
        "KMeansSMOTE":       KMeansSMOTE(random_state=SEED, k_neighbors=2,
                                         cluster_balance_threshold=0.01),
    }
    proposed_variants = {
        # (use_safe, use_entropy, apply_tomek)
        "PROPOSED (Entropy only)":   (False, True,  False),
        "PROPOSED (Entropy+Tomek)":  (False, True,  True),
        "PROPOSED (Safe only)":      (True,  True,  False),
        "PROPOSED (Safe+Tomek)":     (True,  True,  True),
    }
    classifier_names = list(get_classifier_with_tuning.__globals__['get_classifier_with_tuning']
                            .__defaults__) if False else [
        "GBM", "RF", "Bagging", "XGBoost", "ExtraTrees", "AdaBoost", "DT",
        "SVM-RBF", "KNN", "SGD", "LinearSVM", "LogReg", "LDA", "QDA", "GNB"
    ]

    results = []
    fold_idx = 0
    t_start = time.time()

    for train_idx, test_idx in skf.split(X, y):
        fold_idx += 1
        print(f"\n=== Fold {fold_idx}/{n_splits} ===")

        # Train-only preprocessing
        X_tr_raw, X_te_raw = X[train_idx], X[test_idx]
        y_tr,     y_te     = y[train_idx], y[test_idx]

        imp = SimpleImputer(strategy='median').fit(X_tr_raw)
        X_tr_imp = imp.transform(X_tr_raw)
        X_te_imp = imp.transform(X_te_raw)

        scaler = StandardScaler().fit(X_tr_imp)
        X_tr_sc = scaler.transform(X_tr_imp)
        X_te_sc = scaler.transform(X_te_imp)

        pca = PCA(n_components=0.95, random_state=SEED).fit(X_tr_sc)
        X_tr = pca.transform(X_tr_sc)
        X_te = pca.transform(X_te_sc)

        print(f"  PCA dim after fit on training fold: {X_tr.shape[1]}")

        # CNN guidance computed once per fold on training data
        print("  Training CNN topology guide...")
        ent_tr, deep_tr = get_cnn_guidance(X_tr, y_tr)

        # 1) Classical samplers
        for s_name, sampler in samplers_classical.items():
            if sampler is None:
                X_res, y_res = X_tr, y_tr
            else:
                try:
                    X_res, y_res = sampler.fit_resample(X_tr, y_tr)
                except Exception:
                    continue
            _eval_all_clfs(X_res, y_res, X_te, y_te, s_name, classifier_names,
                           fold_idx, results)

        # 2) Proposed variants
        for v_name, (use_safe, use_entropy, apply_tomek) in proposed_variants.items():
            try:
                X_res, y_res = run_ew_ddbs(X_tr, y_tr, ent_tr, deep_tr,
                                           use_safe, use_entropy, apply_tomek)
            except Exception as e:
                print(f"  Error in {v_name}: {str(e)[:60]}")
                continue
            _eval_all_clfs(X_res, y_res, X_te, y_te, v_name, classifier_names,
                           fold_idx, results)

        elapsed = (time.time() - t_start) / 60
        print(f"  Fold {fold_idx} done. Total elapsed: {elapsed:.1f} min")

    df = pd.DataFrame(results)
    df.to_csv('ctu_uhb_experiment_refined.csv', index=False)

    print("\n" + "="*70)
    print("AGGREGATE RESULTS (mean across folds, per Oversampling x Classifier)")
    print("="*70)
    summary = df.groupby(['Oversampling','Classifier']).agg({
        'F1-Macro':         ['mean','std'],
        'G-Mean':           ['mean','std'],
        'BalancedAccuracy': ['mean','std'],
        'AUC':              ['mean','std'],
        'AUPRC':            ['mean','std'],
    }).round(4)
    print(summary.to_string())
    summary.to_csv('ctu_uhb_summary_refined.csv')

    print("\n" + "="*70)
    print("OVERALL F1-MACRO RANKING (mean across all classifiers and folds)")
    print("="*70)
    rank = df.groupby('Oversampling')[['F1-Macro','G-Mean','BalancedAccuracy',
                                       'AUC','AUPRC']].mean().round(4)
    rank = rank.sort_values('F1-Macro', ascending=False)
    print(rank.to_string())

    # Friedman test on F1-Macro using Fold as the block
    try:
        from scipy.stats import friedmanchisquare
        pivot = df.groupby(['Fold','Oversampling'])['F1-Macro'].mean().unstack()
        chi, p = friedmanchisquare(*[pivot[c].values for c in pivot.columns])
        print(f"\nFriedman test (block = Fold, N={pivot.shape[0]}): chi2={chi:.4f}, p={p:.6f}")

        if p < 0.05:
            try:
                import scikit_posthocs as sp
                nem = sp.posthoc_nemenyi_friedman(pivot.values)
                nem.index = pivot.columns
                nem.columns = pivot.columns
                ref = 'PROPOSED (Safe only)' if 'PROPOSED (Safe only)' in nem.columns \
                      else 'PROPOSED (Safe+Tomek)'
                if ref in nem.columns:
                    print(f"\nNemenyi vs {ref}:")
                    pvals = nem[ref].drop(ref).sort_values()
                    for m, pv in pvals.items():
                        sig = "***" if pv<0.001 else ("**" if pv<0.01
                              else ("*" if pv<0.05 else "ns"))
                        print(f"  {m:30s}  p = {pv:.4f}  {sig}")
            except Exception as e:
                print(f"Nemenyi failed: {e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    elapsed = (time.time() - t_start) / 60
    print(f"\nTotal experiment runtime: {elapsed:.1f} min")
    return df


def _eval_all_clfs(X_res, y_res, X_te, y_te, method_name, classifier_names,
                   fold_idx, results):
    """Train+evaluate each classifier (with light tuning) on the resampled fold."""
    for c_name in classifier_names:
        try:
            clf = get_classifier_with_tuning(c_name, X_res, y_res)
            y_pred = clf.predict(X_te)
            y_proba = predict_proba_safe(clf, X_te)
            m = evaluate_model(y_te, y_pred, y_proba)
        except Exception as e:
            m = {'F1-Macro': np.nan, 'BalancedAccuracy': np.nan,
                 'G-Mean': np.nan, 'AUC': np.nan, 'AUPRC': np.nan}
        results.append({
            'Fold': fold_idx,
            'Oversampling': method_name,
            'Classifier': c_name,
            **m,
        })


# =====================================================================
# 7. RUN
# =====================================================================
if __name__ == "__main__":
    print("="*70)
    print("EW-DDBS External Validation on CTU-UHB (METHODOLOGICALLY CONSISTENT)")
    print("="*70)

    # No external CSV needed — fetch records directly from PhysioNet
    X, y = prepare_ctu_uhb(max_records=552)
    df_results = main_experiment(X, y, n_splits=10)


EW-DDBS External Validation on CTU-UHB (METHODOLOGICALLY CONSISTENT)
Fetching record list from PhysioNet (ctu-uhb-ctgdb/1.0.0)...
  -> 552 records available

  ...50/552 processed (scanned 50/552)
  ...100/552 processed (scanned 100/552)
  ...150/552 processed (scanned 150/552)
  ...200/552 processed (scanned 200/552)
  ...250/552 processed (scanned 250/552)
  ...300/552 processed (scanned 300/552)
  ...350/552 processed (scanned 350/552)
  ...400/552 processed (scanned 400/552)
  ...450/552 processed (scanned 450/552)
  ...500/552 processed (scanned 500/552)
  ...550/552 processed (scanned 550/552)

Dataset ready: 552 samples, 30 features
Class distribution: {np.int32(0): np.int64(447), np.int32(1): np.int64(105)}

=== Fold 1/10 ===
  PCA dim after fit on training fold: 13
  Training CNN topology guide...
  Fold 1 done. Total elapsed: 2.8 min

=== Fold 2/10 ===
  PCA dim after fit on training fold: 13
  Training CNN topology guide...
  Fold 2 done. Total elapsed: 5.4 min

=== Fold 3/1

## Wilcoxon paired ablation test
Run this **after** the main experiment finishes. It uses `df_results` from the previous cell.

In [ ]:
# ================== Wilcoxon Signed-Rank Ablation Test ==================
# Paired by (Fold, Classifier) on metrics produced by the main experiment.
# Run AFTER the main experiment above completes (df_results must exist).

from scipy.stats import wilcoxon
import pandas as pd

ABLATION_PAIRS = [
    ("PROPOSED (Entropy only)",  "PROPOSED (Entropy+Tomek)", "F1-Macro"),
    ("PROPOSED (Safe only)",     "PROPOSED (Safe+Tomek)",    "F1-Macro"),
    ("PROPOSED (Entropy only)",  "PROPOSED (Entropy+Tomek)", "AUPRC"),
    ("PROPOSED (Safe only)",     "PROPOSED (Safe+Tomek)",    "AUPRC"),
    ("PROPOSED (Entropy+Tomek)", "PROPOSED (Safe+Tomek)",    "F1-Macro"),
    ("PROPOSED (Entropy+Tomek)", "PROPOSED (Safe+Tomek)",    "AUPRC"),
]

print("=== Wilcoxon signed-rank tests for CTU-UHB ablation pairs ===")
print("(paired by Fold x Classifier; expected N = 10 folds x 15 classifiers = 150)\n")

rows = []
for a, b, m in ABLATION_PAIRS:
    sub_a = (df_results[df_results['Oversampling'] == a]
             .set_index(['Fold','Classifier'])[m].sort_index())
    sub_b = (df_results[df_results['Oversampling'] == b]
             .set_index(['Fold','Classifier'])[m].sort_index())
    common = sub_a.index.intersection(sub_b.index)
    sub_a = sub_a.loc[common].dropna()
    sub_b = sub_b.loc[common].dropna()
    common = sub_a.index.intersection(sub_b.index)
    sub_a = sub_a.loc[common]
    sub_b = sub_b.loc[common]

    diff = sub_b.values - sub_a.values
    median_diff = float(pd.Series(diff).median())
    mean_diff   = float(pd.Series(diff).mean())

    if (diff == 0).all() or len(diff) < 1:
        stat, p = float('nan'), float('nan')
    else:
        stat, p = wilcoxon(sub_a.values, sub_b.values,
                           zero_method='wilcox', alternative='two-sided')

    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    rows.append({
        'metric':   m,
        'A':        a,
        'B':        b,
        'n_pairs':  len(diff),
        'median_diff_B_minus_A': round(median_diff, 4),
        'mean_diff_B_minus_A':   round(mean_diff, 4),
        'wilcoxon_stat': None if pd.isna(stat) else round(stat, 3),
        'p_value':       None if pd.isna(p)    else round(p, 4),
        'sig_alpha_0.05': sig,
    })

wilcoxon_df = pd.DataFrame(rows)
print(wilcoxon_df.to_string(index=False))
wilcoxon_df.to_csv("ctu_uhb_ablation_wilcoxon.csv", index=False)
print("\nSaved to ctu_uhb_ablation_wilcoxon.csv")


=== Wilcoxon signed-rank tests for CTU-UHB ablation pairs ===
(paired by Fold x Classifier; expected N = 10 folds x 15 classifiers = 150)

  metric                        A                        B  n_pairs  median_diff_B_minus_A  mean_diff_B_minus_A  wilcoxon_stat  p_value sig_alpha_0.05
F1-Macro  PROPOSED (Entropy only) PROPOSED (Entropy+Tomek)      150                 0.0000              -0.0050         4166.0   0.3517             ns
F1-Macro     PROPOSED (Safe only)    PROPOSED (Safe+Tomek)      150                -0.0027               0.0013         4450.0   0.8721             ns
   AUPRC  PROPOSED (Entropy only) PROPOSED (Entropy+Tomek)      150                -0.0049              -0.0068         4911.0   0.1585             ns
   AUPRC     PROPOSED (Safe only)    PROPOSED (Safe+Tomek)      150                 0.0022               0.0065         4984.0   0.2030             ns
F1-Macro PROPOSED (Entropy+Tomek)    PROPOSED (Safe+Tomek)      150                 0.0038               0

## Download outputs
After the runs complete, the following CSVs will be in the working directory:
- `ctu_uhb_experiment_refined.csv` — per fold × method × classifier
- `ctu_uhb_summary_refined.csv` — aggregated (mean ± std)
- `ctu_uhb_ablation_wilcoxon.csv` — paired ablation tests

In [ ]:
# Convenience: download CSVs to your machine (Colab only)
try:
    from google.colab import files
    for f in ['ctu_uhb_experiment_refined.csv',
             'ctu_uhb_summary_refined.csv',
             'ctu_uhb_ablation_wilcoxon.csv']:
        try:
            files.download(f)
        except Exception as e:
            print(f'Could not download {f}: {e}')
except ImportError:
    print('Not in Colab — CSVs are saved in the working directory.')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>